# AGROP-42 — Comparación experimental condicionada

**Estado inicial:** las descripciones aceptadas no son etiquetas de calidad. Este cuaderno **no entrena** mientras no se acredite identidad de pieza, sesión, cobertura y clase E0/E1/E2 para mango y tomate. No interpreta color como enfermedad o madurez interna.

Requisitos mínimos (por producto): ≥3 grupos independientes (unidos por pieza **o** sesión), ≥2 clases verificadas, máscaras revisadas, y particiones separadas por grupo. Los resultados solo se registran cuando hay al menos un split con ambas clases en entrenamiento y prueba; no se informan resultados basados en datos de ejemplo.


In [ ]:
from pathlib import Path
from collections import Counter
import csv, json
from vision.evaluation.agrop42.auditar_revisiones import auditar_csv, asignar_grupos, _fila_elegible

csv_path = Path("vision/evaluation/agrop42/AGROP42_revisiones_texto_aprobado_v2.csv")
informe = auditar_csv(csv_path)
print(json.dumps(informe, ensure_ascii=False, indent=2))
HABILITADO = informe["estado_global"] == "apto_para_diseno_experimental"
print("Experimento habilitado:", HABILITADO)


## Fuente de atributos (solo después de acreditar el gate)

Preparar un CSV independiente `vision/experiments/AGROP42/atributos_validados.csv` con columnas `registro_id`, `r_mediana`, `g_mediana`, `b_mediana`, `area_relativa`, `mascara_confirmada`, `evidencia_mascara`. Los valores RGB corresponden a píxeles de una máscara *confirmada*, no simplemente propuesta. `area_relativa` es la proporción del área analizada. Una máscara provisional **no** es suficiente. No mezclar especies en un modelo.

**Enfoque A:** `DummyClassifier` (clase mayoritaria), referencia. **Enfoque B:** `RandomForestClassifier` fijo (`n_estimators=100`, `random_state=42`, `class_weight="balanced"`). Métricas por producto: exactitud, exactitud equilibrada y F1 macro; grupos separados para evitar filtración. Estos enfoques son propuestas, no modelos entrenados.


In [ ]:
features_path = Path("vision/experiments/AGROP42/atributos_validados.csv")
ATRIBUTOS = ("r_mediana", "g_mediana", "b_mediana", "area_relativa")
if not HABILITADO:
    print("BLOQUEADO: auditoría sin suficientes grupos y clases acreditadas. No hay entrenamiento ni métricas.")
elif not features_path.is_file():
    print("BLOQUEADO: no existe un CSV de atributos con máscaras confirmadas. No crear datos ficticios.")
else:
    print("Gate documental aprobado y CSV presente: listo para validar características y particiones.")


In [ ]:
resultados = []
if HABILITADO and features_path.is_file():
    from sklearn.dummy import DummyClassifier
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
    from sklearn.model_selection import GroupShuffleSplit
    with csv_path.open(encoding="utf-8-sig", newline="") as f:
        anotaciones = list(csv.DictReader(f))
    with features_path.open(encoding="utf-8-sig", newline="") as f:
        features_list = list(csv.DictReader(f))
    assert len({r["registro_id"] for r in features_list}) == len(features_list), "ID de atributos repetido"
    features = {r["registro_id"]:r for r in features_list}
    for producto in ("mango", "tomate"):
        filas = [r for r in anotaciones if r["producto"] == producto and _fila_elegible(r)]
        grupos = asignar_grupos(filas)
        X, y, g = [], [], []
        for fila in filas:
            rid = fila["registro_id"]
            feat = features.get(rid)
            if feat is None or feat.get("mascara_confirmada", "").lower() not in ("true", "si", "sí", "1") or not feat.get("evidencia_mascara", "").strip():
                raise ValueError(f"{producto}/{rid}: máscara no acreditada o atributos ausentes")
            valores = [float(feat[c]) for c in ATRIBUTOS]
            if not all(0 <= z <= 255 for z in valores[:3]) or not (0 <= valores[3] <= 1):
                raise ValueError(f"{producto}/{rid}: atributos fuera de rango")
            X.append(valores); y.append(fila["etiqueta_visual"]); g.append(grupos[rid])
        if len(set(g)) < 3 or len(set(y)) < 2:
            print(producto, ": BLOQUEADO después del cruce con atributos")
            continue
        splitter = GroupShuffleSplit(n_splits=20, test_size=0.33, random_state=42)
        folds_validos = 0
        for idx, (train, test) in enumerate(splitter.split(X, y, g)):
            y_train=[y[i] for i in train]; y_test=[y[i] for i in test]
            if len(set(y_train)) < 2 or len(set(y_test)) < 2:
                continue
            assert {g[i] for i in train}.isdisjoint({g[i] for i in test})
            folds_validos += 1
            for enfoque, modelo in [
                ("Referencia - mayoría", DummyClassifier(strategy="most_frequent")),
                ("Bosque aleatorio", RandomForestClassifier(n_estimators=100, random_state=42, class_weight="balanced")),
            ]:
                modelo.fit([X[i] for i in train], y_train)
                preds=modelo.predict([X[i] for i in test])
                resultados.append({"producto":producto, "fold":idx, "enfoque":enfoque,
                    "train":len(train), "test":len(test),
                    "accuracy":accuracy_score(y_test,preds),
                    "balanced_accuracy":balanced_accuracy_score(y_test,preds),
                    "f1_macro":f1_score(y_test,preds,average="macro",zero_division=0)})
        if not folds_validos:
            print(producto, ": BLOQUEADO: no existen particiones que preserven ambas clases en train y test")
    if resultados:
        print(json.dumps(resultados,ensure_ascii=False,indent=2))
    else:
        print("Sin resultados: mantener AGROP-42 como evaluación pendiente")
else:
    print("Ninguna métrica calculada.")


## Interpretación

La prueba sintética del código no acredita rendimiento con frutas reales. Para reportar resultados se requieren splits viables independientes, imágenes por pieza y sesión, rúbricas revisadas, auditoría de calidad, máscaras confirmadas y bitácora con versión de código/dataset. No reportar métricas cuando `resultados` es una lista vacía.
